In [ ]:
import os
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -----------------------------
# 1. Dataset
# -----------------------------
class CarDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform
        self.tabular_data = df.drop(columns=['path', 'price']).values.astype(float)
        self.targets = df['price'].values.astype(float)

        # Нормализация табличных данных
        self.scaler = StandardScaler()
        self.tabular_data = self.scaler.fit_transform(self.tabular_data)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row['path'])
        image = transforms.functional.pil_to_tensor(transforms.functional.pil_image_loader(img_path)).float() / 255.0

        if self.transform:
            image = self.transform(image)

        tabular = torch.tensor(self.tabular_data[idx], dtype=torch.float32)
        target = torch.tensor(self.targets[idx], dtype=torch.float32)
        return image, tabular, target

# -----------------------------
# 2. Model
# -----------------------------
class MultiModalModel(nn.Module):
    def __init__(self, tabular_input_size, hidden_dim=128):
        super().__init__()
        # Предобученная CNN для изображений
        self.cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.cnn.fc = nn.Identity()  # Убираем последний классификационный слой

        # Полносвязная сеть для табличных данных
        self.tabular_fc = nn.Sequential(
            nn.Linear(tabular_input_size, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        # Общий слой для объединения признаков
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim + 512, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)  # Регрессия
        )

    def forward(self, image, tabular):
        img_feat = self.cnn(image)
        tab_feat = self.tabular_fc(tabular)
        combined = torch.cat([img_feat, tab_feat], dim=1)
        out = self.fc(combined)
        return out.squeeze(1)

# -----------------------------
# 3. PyTorch Lightning Module
# -----------------------------
class CarPricePredictor(pl.LightningModule):
    def __init__(self, tabular_input_size, lr=1e-3):
        super().__init__()
        self.model = MultiModalModel(tabular_input_size)
        self.criterion = nn.MSELoss()
        self.lr = lr

    def forward(self, image, tabular):
        return self.model(image, tabular)

    def training_step(self, batch, batch_idx):
        image, tabular, target = batch
        pred = self(image, tabular)
        loss = self.criterion(pred, target)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        image, tabular, target = batch
        pred = self(image, tabular)
        loss = self.criterion(pred, target)
        self.log("val_loss", loss)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

# -----------------------------
# 4. Data preparation
# -----------------------------
df = pd.read_csv("data.csv")  # ваша таблица
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = CarDataset(train_df, image_dir="images", transform=transform)
val_dataset = CarDataset(val_df, image_dir="images", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

# -----------------------------
# 5. Training
# -----------------------------
tabular_input_size = train_dataset.tabular_data.shape[1]
model = CarPricePredictor(tabular_input_size)

trainer = pl.Trainer(max_epochs=10, accelerator="auto", devices=1)
trainer.fit(model, train_loader, val_loader)

In [ ]:
#  self.cnn = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

In [ ]:
# ✅ FiLM позволяет табличным признакам направлять визуальные признаки, что часто улучшает мульти-модальные задачи.

# class FiLMFusionModel(nn.Module):
#     def __init__(self, tabular_input_size, hidden_dim=128):
#         super().__init__()
#         self.cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
#         self.cnn.fc = nn.Identity()  # 512 features

#         # FiLM генератор: из табличных данных -> gamma и beta для визуального эмбеддинга
#         self.film_gen = nn.Sequential(
#             nn.Linear(tabular_input_size, hidden_dim),
#             nn.ReLU(),
#             nn.Linear(hidden_dim, 512*2)  # gamma и beta
#         )

#         self.fc = nn.Sequential(
#             nn.Linear(512, hidden_dim),
#             nn.ReLU(),
#             nn.Linear(hidden_dim, 1)
#         )

#     def forward(self, image, tabular):
#         img_feat = self.cnn(image)  # [batch, 512]
#         gamma_beta = self.film_gen(tabular)  # [batch, 1024]
#         gamma, beta = gamma_beta.chunk(2, dim=1)
#         img_feat = img_feat * (1 + gamma) + beta  # FiLM
#         out = self.fc(img_feat)
#         return out.squeeze(1)


🔹 Особенности этого pipeline:

Аугментации: горизонтальное отражение, цветовые джиттеры, вращение — повышают устойчивость модели.

FiLM fusion: табличные данные влияют на визуальные признаки.

ResNet50 backbone: сильнее ResNet18, 2048 признаков.

Log-transform таргета: помогает при скошенном распределении цены.

K-Fold CV: надежная оценка качества и уменьшение variance.

LR scheduler: CosineAnnealingLR для плавного уменьшения lr.

In [ ]:
import os
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
import pytorch_lightning as pl
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import numpy as np

# -----------------------------
# Dataset
# -----------------------------
class CarDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.tabular_data = df.drop(columns=['path', 'price']).values.astype(float)
        self.targets = np.log1p(df['price'].values.astype(float))  # лог-трансформация

        # нормализация табличных данных
        self.scaler = StandardScaler()
        self.tabular_data = self.scaler.fit_transform(self.tabular_data)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row['path'])
        image = transforms.functional.pil_to_tensor(transforms.functional.pil_image_loader(img_path)).float() / 255.0

        if self.transform:
            image = self.transform(image)

        tabular = torch.tensor(self.tabular_data[idx], dtype=torch.float32)
        target = torch.tensor(self.targets[idx], dtype=torch.float32)
        return image, tabular, target

# -----------------------------
# FiLM Fusion Model
# -----------------------------
class FiLMFusionModel(nn.Module):
    def __init__(self, tabular_input_size, hidden_dim=128):
        super().__init__()
        self.cnn = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.cnn.fc = nn.Identity()  # 2048 features

        # FiLM генератор
        self.film_gen = nn.Sequential(
            nn.Linear(tabular_input_size, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2048*2)
        )

        self.fc = nn.Sequential(
            nn.Linear(2048, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, image, tabular):
        img_feat = self.cnn(image)
        gamma_beta = self.film_gen(tabular)
        gamma, beta = gamma_beta.chunk(2, dim=1)
        img_feat = img_feat * (1 + gamma) + beta  # FiLM
        out = self.fc(img_feat)
        return out.squeeze(1)

# -----------------------------
# Lightning Module
# -----------------------------
class CarPricePredictor(pl.LightningModule):
    def __init__(self, tabular_input_size, lr=1e-3):
        super().__init__()
        self.model = FiLMFusionModel(tabular_input_size)
        self.criterion = nn.MSELoss()
        self.lr = lr

    def forward(self, image, tabular):
        return self.model(image, tabular)

    def training_step(self, batch, batch_idx):
        image, tabular, target = batch
        pred = self(image, tabular)
        loss = self.criterion(pred, target)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        image, tabular, target = batch
        pred = self(image, tabular)
        loss = self.criterion(pred, target)
        self.log("val_loss", loss)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        return [optimizer], [scheduler]

# -----------------------------
# Data preparation & augmentations
# -----------------------------
df = pd.read_csv("data.csv")  # Ваша таблица
image_dir = "images"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomRotation(10),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

dataset = CarDataset(df, image_dir=image_dir, transform=transform)

# -----------------------------
# K-Fold Cross Validation
# -----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1

for train_idx, val_idx in kf.split(dataset):
    print(f"Training fold {fold}")
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_subset, batch_size=16, shuffle=False, num_workers=4)

    tabular_input_size = dataset.tabular_data.shape[1]
    model = CarPricePredictor(tabular_input_size)

    trainer = pl.Trainer(max_epochs=20, accelerator="auto", devices=1)
    trainer.fit(model, train_loader, val_loader)

    fold += 1
